# CLASSIFICATION MODELING

**IMPORTANTE**: Recuerda hacer una copia de esta plantilla para no machacar la original.

**IMPORTANTE**: Esta plantilla está diseñada para una visión de máximos utilizando el framework de The Ultimate Algo Machine. Si tienes problemas de memoria o rendimiento recuerda reducir el problema mediante:

* Muestreo
* Balanceo undersampling
* Reducir el número de algoritmos a testar
* Reducir el número de parámetros a testar
* Usar random search y especificar un n_iter adecuado

## IMPORT PACKAGES

In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
%matplotlib inline

from sklearn.base import clone
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from sklearn.metrics import roc_curve, roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

#from sklearn.metrics import plot_precision_recall_curve
from sklearn.metrics import PrecisionRecallDisplay

#from yellowbrick.classifier import discrimination_threshold

# Autocomplete settings
%config IPCompleter.greedy=True

# Disable scientific notation for pandas
pd.options.display.float_format = '{:.2f}'.format

# Disable warnings to keep output clean
import warnings
warnings.filterwarnings("ignore")

## IMPORT DATA

In [3]:
project_path = Path('/Users/gloriadelriomarquez/Documents/Career/Projects/ml-lab/client_churn')
data_path = project_path / "02_data" / "train"

X_file_name = "X_selected.pickle"
y_file_name = "y_selected.pickle"

X = pd.read_pickle(data_path / X_file_name)
y = pd.read_pickle(data_path / y_file_name)

## MODELING

In [4]:
# Create train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

In [5]:
from sklearn.pipeline import Pipeline

# Create the pipeline with a placeholder model (overwritten by the grid)
pipe = Pipeline([('model', RandomForestClassifier())])

# Dictionary of models and parameters to test
param_grid = [
    # Logistic Regression with L1 / L2 regularization
    {
        'model': [LogisticRegression(solver='saga', max_iter=2000, random_state=42)],
        'model__penalty': ['l1', 'l2'],
        'model__C': [0.01, 0.1, 1, 10]
    },

    # Logistic Regression with ElasticNet regularization
    {
        'model': [LogisticRegression(solver='saga', penalty='elasticnet', max_iter=2000, random_state=42)],
        'model__C': [0.1, 1, 10],
        'model__l1_ratio': [0.25, 0.5, 0.75]
    },

    # Random Forest
    {
        'model': [RandomForestClassifier(n_jobs=-1, random_state=42)],
        'model__max_depth': [10, 20],
        'model__min_samples_leaf': [500],
        'model__n_estimators': [100, 200]
    },

    # XGBoost
    {
        'model': [XGBClassifier(n_jobs=-1, tree_method='hist', verbosity=0, random_state=42)],
        'model__learning_rate': [0.05, 0.1],
        'model__max_depth': [4, 6],
        'model__min_child_weight': [1, 10],
        'model__reg_lambda': [1, 10],
        'model__n_estimators': [300, 600]
    },

    # Histogram Gradient Boosting
    {
        'model': [HistGradientBoostingClassifier(
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            max_iter=1000,
            random_state=42
        )],
        'model__learning_rate': [0.05, 0.1],
        'model__max_leaf_nodes': [15, 31, 63],
        'model__min_samples_leaf': [200, 500],
        'model__l2_regularization': [0, 1, 10]
    }
]

### Hyperparameter optimization with Grid Search

In [7]:
import time

grid_search = GridSearchCV(
                estimator=pipe,
                param_grid=param_grid,
                cv=3,
                scoring='roc_auc',
                return_train_score=True,
                verbose=0,
                n_jobs=-1
               )

start = time.time()

model = grid_search.fit(X_train, y_train)

elapsed = time.time() - start
print(f"Tiempo total: {elapsed / 60:.1f} minutos")

pd.DataFrame(grid_search.cv_results_).sort_values(by='rank_test_score')

Tiempo total: 143.4 minutos


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model,param_model__C,param_model__penalty,param_model__l1_ratio,param_model__max_depth,param_model__min_samples_leaf,...,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,mean_train_score,std_train_score
57,64.90,2.95,4.42,0.30,HistGradientBoostingClassifier(early_stopping=...,NaN,NaN,NaN,NaN,200.00,...,0.92,0.92,0.92,0.00,1,0.93,0.93,0.93,0.93,0.00
69,71.11,6.36,4.71,0.42,HistGradientBoostingClassifier(early_stopping=...,NaN,NaN,NaN,NaN,200.00,...,0.92,0.92,0.92,0.00,2,0.93,0.93,0.93,0.93,0.00
75,37.29,0.91,2.43,0.03,HistGradientBoostingClassifier(early_stopping=...,NaN,NaN,NaN,NaN,200.00,...,0.92,0.92,0.92,0.00,3,0.93,0.93,0.93,0.93,0.00
81,66.86,3.45,4.87,0.23,HistGradientBoostingClassifier(early_stopping=...,NaN,NaN,NaN,NaN,200.00,...,0.92,0.92,0.92,0.00,4,0.92,0.93,0.92,0.93,0.00
63,39.23,2.65,2.62,0.20,HistGradientBoostingClassifier(early_stopping=...,NaN,NaN,NaN,NaN,200.00,...,0.92,0.92,0.92,0.00,5,0.93,0.93,0.93,0.93,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9,1167.54,13.01,0.13,0.01,"LogisticRegression(max_iter=2000, penalty='ela...",0.10,NaN,0.50,NaN,NaN,...,0.85,0.85,0.85,0.00,85,0.85,0.85,0.85,0.85,0.00
4,1127.59,8.11,0.12,0.00,"LogisticRegression(max_iter=2000, random_state...",1.00,l1,NaN,NaN,NaN,...,0.85,0.85,0.85,0.00,86,0.85,0.85,0.85,0.85,0.00
10,1169.22,17.86,0.13,0.00,"LogisticRegression(max_iter=2000, penalty='ela...",0.10,NaN,0.75,NaN,NaN,...,0.85,0.85,0.85,0.00,87,0.85,0.85,0.85,0.85,0.00
2,1189.80,27.85,0.15,0.02,"LogisticRegression(max_iter=2000, random_state...",0.10,l1,NaN,NaN,NaN,...,0.85,0.85,0.85,0.00,88,0.85,0.85,0.85,0.85,0.00


In [28]:
pd.set_option('display.float_format', '{:.4f}'.format)

# Sustituye grid_search por el nombre de tu GridSearchCV
print("Mejor ROC-AUC CV:", grid_search.best_score_)
print("\nMejores hiperparámetros:")
print(grid_search.best_params_)

print("\nMejor estimador:")
print(grid_search.best_estimator_)

# Comparación de los 10 mejores resultados
cols = [
    "param_model",
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "mean_fit_time",
    "rank_test_score"
]

display(
    pd.DataFrame(grid_search.cv_results_)
    .sort_values("rank_test_score")
    [cols]
    .head(10)
)


Mejor ROC-AUC CV: 0.9166244700469721

Mejores hiperparámetros:
{'model': HistGradientBoostingClassifier(early_stopping=True, l2_regularization=0,
                               max_iter=1000, max_leaf_nodes=63,
                               min_samples_leaf=200, n_iter_no_change=20,
                               random_state=42), 'model__l2_regularization': 0, 'model__learning_rate': 0.05, 'model__max_leaf_nodes': 63, 'model__min_samples_leaf': 200}

Mejor estimador:
Pipeline(steps=[('model',
                 HistGradientBoostingClassifier(early_stopping=True,
                                                l2_regularization=0,
                                                learning_rate=0.05,
                                                max_iter=1000,
                                                max_leaf_nodes=63,
                                                min_samples_leaf=200,
                                                n_iter_no_change=20,
                         

,param_model,mean_test_score,std_test_score,mean_train_score,mean_fit_time,rank_test_score
57,HistGradientBoostingClassifier(early_stopping=...,0.9166,0.0010,0.9282,64.8953,1
69,HistGradientBoostingClassifier(early_stopping=...,0.9166,0.0011,0.9284,71.1144,2
75,HistGradientBoostingClassifier(early_stopping=...,0.9165,0.0011,0.9276,37.2934,3
81,HistGradientBoostingClassifier(early_stopping=...,0.9164,0.0010,0.9251,66.8612,4
63,HistGradientBoostingClassifier(early_stopping=...,0.9164,0.0009,0.9299,39.2338,5
87,HistGradientBoostingClassifier(early_stopping=...,0.9163,0.0009,0.9279,42.4925,6
55,HistGradientBoostingClassifier(early_stopping=...,0.9162,0.0008,0.9253,81.0490,7
67,HistGradientBoostingClassifier(early_stopping=...,0.9162,0.0009,0.9245,74.5573,8
73,HistGradientBoostingClassifier(early_stopping=...,0.9162,0.0007,0.9266,47.7510,9
46,"XGBClassifier(base_score=None, booster=None, c...",0.9161,0.0009,0.9261,24.8281,10


### Conclusiones de la búsqueda de hiperparámetros

1. **No hay overfitting.** AUC de train (0,93) y validación (0,92) muy próximos, con desviación entre folds casi nula.
2. **Los hiperparámetros apenas importan.** Las 10 mejores combinaciones (incluida XGBoost) redondean a 0,92: el rendimiento lo limitan las variables, no el ajuste fino.
3. **Selección del quinto modelo por eficiencia computacional.** Se selecciona la configuración de HistGradientBoostingClassifier situada en la quinta posición del ranking, priorizando el equilibrio entre rendimiento predictivo y coste de entrenamiento. Su AUC es muy próximo al de la mejor configuración, pero su tiempo medio de entrenamiento es de 39,23 segundos frente a 64,90 segundos, lo que supone una reducción aproximada del 40 %. Los tiempos corresponden a las mediciones realizadas durante la validación cruzada y pueden variar en otras ejecuciones.

#### Mejoras para futuros experimentos

| Modelo | Ajuste | Motivo |
|---|---|---|
| Logistic Regression L2 | `solver='lbfgs'` | Más eficiente para L2 |
| Logistic Regression L1 / Elastic Net | `solver='saga'` | Único compatible |
| Random Forest | Mantener | Espacio de búsqueda pequeño |
| XGBoost | Menos combinaciones o `RandomizedSearchCV` | 32 configuraciones |
| HistGradientBoosting | Afinar alrededor de las mejores | Mejores resultados |

- **`penalty` está deprecado en scikit-learn 1.8.** Usar `l1_ratio` en su lugar: `0` → L2, `1` → L1, intermedio → Elastic Net.
- **SAGA es lento con L1 y Elastic Net**, sobre todo con muchas observaciones y variables correlacionadas. Revisar si alcanza `max_iter=2000`: si es así, no ha convergido y bajar el límite no lo soluciona.

In [31]:
# Resultados del GridSearchCV
results = pd.DataFrame(grid_search.cv_results_)

# Seleccionar el modelo con ranking 5
selected = results.loc[results["rank_test_score"] == 5].iloc[0]

# Recuperar su configuración
print("ROC-AUC CV:", selected["mean_test_score"])
print("Tiempo medio:", selected["mean_fit_time"])
print("\nParámetros:\n", selected["params"])

# Construir el pipeline con la configuración seleccionada
model_5 = clone(pipe).set_params(**selected["params"])

# Entrenar con todos los datos de entrenamiento
model_5.fit(X_train, y_train)

# Obtener probabilidades
pred_model_5 = model_5.predict_proba(X_test)[:, 1]

ROC-AUC CV: 0.9164103265921136
Tiempo medio: 39.233810345331825

Parámetros:
 {'model': HistGradientBoostingClassifier(early_stopping=True, l2_regularization=0,
                               max_iter=1000, max_leaf_nodes=63,
                               min_samples_leaf=200, n_iter_no_change=20,
                               random_state=42), 'model__l2_regularization': 0, 'model__learning_rate': 0.1, 'model__max_leaf_nodes': 63, 'model__min_samples_leaf': 200}


## EVALUATION

In [32]:
# Predict on the test set (probabilities)
y_proba = model_5.predict_proba(X_test)[:, 1]

# Evaluate ROC AUC
roc_auc_score(y_test, y_proba)

0.9165076868110751

In [35]:
# Inspect the model
model_5

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[float64](2,)","[0.,1.]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](79,)","['last_day_all_auto_renew_1.0','last_day_any_cancel_0.0', 'last_day_max_payment_rs',...,'last_day_all_auto_renew_2.0','city_19.0', 'has_multi_non_cancelled_day_2.0']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,79
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",1000
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",63
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",200


## REPORTING DEL MODELO

### Gain Chart

In [16]:
import numpy as np
import pandas as pd
import plotly.express as px

def cumulative_gain(y_true, y_scores):
    """
    Devuelve:
      - percent: % de clientes ordenados por score (0–1)
      - gains: % acumulado de positivos capturados (0–1)
    """
    y_true = np.asarray(y_true)
    y_scores = np.asarray(y_scores)

    # Ordenar por probabilidad descendente
    sorted_idx = np.argsort(y_scores)[::-1]
    y_true_sorted = y_true[sorted_idx]

    total_positives = y_true.sum()
    if total_positives == 0:
        raise ValueError("No hay positivos en y_true.")

    gains = np.cumsum(y_true_sorted) / total_positives
    percent = np.arange(1, len(y_true_sorted) + 1) / len(y_true_sorted)

    return percent, gains


def plot_gain_chart(y_true, y_scores, title="Gráfico de Ganancias (Gain Chart)"):
    # 1) Calcular curva
    percent, gains = cumulative_gain(y_true, y_scores)

    # 2) DataFrame para Plotly
    df_gain = pd.DataFrame({
        "percent": percent,
        "Modelo": gains,
        "Aleatorio": percent
    })

    # 3) Pasar a formato largo para tener 2 líneas
    df_long = df_gain.melt(
        id_vars="percent",
        value_vars=["Modelo", "Aleatorio"],
        var_name="Curva",
        value_name="gains"
    )

    # 4) Gráfico
    fig = px.line(
        df_long,
        x="percent",
        y="gains",
        color="Curva",
        labels={
            "percent": "Porcentaje de clientes ordenados por score",
            "gains": "Porcentaje acumulado de positivos capturados",
        },
        title=title,
    )

    fig.update_layout(
        template="plotly_white",
        legend_title="",
    )

    fig.show(renderer="iframe")
    return fig

In [36]:
y_scores = model_5.predict_proba(X_test)[:, 1]
gain_chart = plot_gain_chart(y_test, y_scores)

### Lift Chart

In [39]:
import plotly.express as px

def cumulative_gain_curve(y_true, y_scores):
    """
    Cálculo de Curva de Ganancias Acumuladas (para Lift Chart)
    """
    y_true = np.asarray(y_true)
    y_scores = np.asarray(y_scores)

    sorted_idx = np.argsort(y_scores)[::-1]
    y_sorted = y_true[sorted_idx]

    total_pos = y_true.sum()
    gains = np.cumsum(y_sorted) / total_pos
    percent = np.arange(1, len(y_sorted) + 1) / len(y_sorted)

    return percent, gains


def lift_chart(y_true, modelos_scores, incluir_baseline=True):
    """
    Genera un Lift Chart en Plotly Express.

    Parámetros
    ----------
    y_true : array-like
        Etiquetas reales 0/1
    modelos_scores : dict
        Diccionario con formato:
        {
            "Nombre Modelo 1": scores1, 
            "Nombre Modelo 2": scores2,
            ...
        }
    incluir_baseline : bool
        Si True, añade la línea aleatoria (lift = 1)
    """

    df_lift = [ ]

    for nombre, scores in modelos_scores.items():
        p, g = cumulative_gain_curve(y_true, scores)
        lift = g / p

        df_temp = pd.DataFrame({
            "percent": p,
            "lift": lift,
            "Modelo": nombre
        })
        df_lift.append(df_temp)

    # Baseline aleatoria
    if incluir_baseline:
        percent = np.linspace(0.01, 1, 100)
        df_lift.append(pd.DataFrame({
            "percent": percent,
            "lift": np.ones_like(percent),
            "Modelo": "Aleatorio"
        }))

    df_lift = pd.concat(df_lift, ignore_index=True)

    # Gráfico
    fig = px.line(
        df_lift,
        x="percent",
        y="lift",
        color="Modelo",
        title="Lift Chart",
        labels={
            "percent": "% de clientes contactados",
            "lift": "Lift acumulado"
        },
    )

    fig.update_layout(template="plotly_white", height=600, width=900)
    fig.show(renderer="iframe")
    return fig

In [40]:

fig_lift = lift_chart(
    y_true=y_test,
    modelos_scores={
        "HistGradientBoosting": model_5.predict_proba(X_test)[:, 1]
    }
)


### ROC Chart

In [41]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import roc_curve, auc

y_proba = model.best_estimator_.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

df_roc = pd.DataFrame({
    "FPR": fpr,
    "TPR": tpr
})

fig = px.line(
    df_roc,
    x="FPR",
    y="TPR",
    title=f"ROC Curve (AUC = {roc_auc:.3f})"
)

# Random guess line
fig.add_shape(
    type="line",
    x0=0, y0=0,
    x1=1, y1=1,
    line=dict(color="gray", dash="dash")
)

fig.update_layout(
    template="plotly_white",
    height=500,
    width=1000,
    font=dict(size=9),
    title_x=0.5,
    title_y=0.8
)

fig.show(renderer="iframe")

## UMBRAL DE CORTE

| Criterio | Qué hace | Cuándo usarlo |
|---|---|---|
| **Youden (J)** | Maximiza `TPR − FPR` en la curva ROC | Cuando importan por igual los falsos positivos y los falsos negativos |
| **Máximo F1** | Equilibra precisión y recall | Clases desbalanceadas; el más habitual en churn |
| **Coste de negocio** | Minimiza el coste de los errores (perder un cliente vs. hacer una oferta innecesaria) | Si se conocen esos costes; es el más defendible |
| **Capacidad** | Selecciona el top N % de clientes con mayor probabilidad | Si el equipo solo puede contactar a X clientes |

### Precision-Recall chart 

In [46]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import precision_recall_curve, average_precision_score

def plot_precision_recall_models(models, X, y, positive_label=1,
                                 title="Curvas Precision–Recall - Modelos"):
    """
    Dibuja curvas Precision–Recall de varios modelos con Plotly Express.

    Parámetros
    ----------
    models : dict
        {"Nombre modelo": modelo_entrenado}
    X : array-like
        Variables predictoras.
    y : array-like
        Etiquetas reales (0/1).
    positive_label : int/str
        Valor de la clase positiva.
    """
    y = np.asarray(y)
    df_list = []

    for name, model in models.items():
        # 1) Scores
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X)
            pos_idx = list(model.classes_).index(positive_label)
            y_scores = proba[:, pos_idx]
        elif hasattr(model, "decision_function"):
            y_scores = model.decision_function(X)
        else:
            y_scores = model.predict(X)

        # 2) Curva PR
        precision, recall, _ = precision_recall_curve(y, y_scores, pos_label=positive_label)

        # 3) Average Precision (AUC-PR)
        ap = average_precision_score(y, y_scores)

        df_pr = pd.DataFrame({
            "Recall": recall,
            "Precision": precision,
            "Modelo": f"{name} (AP={ap:.3f})"
        })
        df_list.append(df_pr)

    df_pr_all = pd.concat(df_list, ignore_index=True)

    fig = px.line(
        df_pr_all,
        x="Recall",
        y="Precision",
        color="Modelo",
        title=title,
        labels={"Recall": "Recall", "Precision": "Precision"}
    )

    fig.update_layout(
        template="plotly_white",
        height=500,
        width=900,
        legend_title_text=""
    )

    fig.show(renderer="iframe")
    return fig

In [47]:
fig_pr = plot_precision_recall_models(
    models={"HistGradientBoosting": model_5},
    X=X_test,
    y=y_test
)

### Yellowbrick chart 

Máximo F1: umbral más alto, más equilibrio entre precisión y recall.

In [48]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import precision_score, recall_score, f1_score

def plot_threshold_curve(
    model, X, y, positive_label=1,
    include_queue_rate=False,
    title="Gráfico de umbrales (Precision / Recall / F1)",
    step=0.05
):
    """
    Gráfico de métricas vs umbral para un modelo binario.
    """

    y = np.asarray(y)

    # 1) Scores continuos del modelo
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        classes_ = list(model.classes_)
        if positive_label not in classes_:
            raise ValueError(f"La clase positiva {positive_label} no está en model.classes_ = {classes_}")
        pos_idx = classes_.index(positive_label)
        scores = proba[:, pos_idx]
        thr_min, thr_max = 0.0, 1.0   # rango natural de probas

    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        thr_min, thr_max = scores.min(), scores.max()

    else:
        raise ValueError("El modelo necesita predict_proba o decision_function para este gráfico.")

    # 2) Rejilla de thresholds en el rango de scores
    thresholds = np.arange(thr_min, thr_max + step, step)

    precisions, recalls, f1s, queue_rates = [], [], [], []

    for thr in thresholds:
        y_pred = (scores >= thr).astype(int)

        precisions.append(precision_score(y, y_pred, pos_label=positive_label, zero_division=0))
        recalls.append(recall_score(y, y_pred, pos_label=positive_label, zero_division=0))
        f1s.append(f1_score(y, y_pred, pos_label=positive_label, zero_division=0))
        queue_rates.append(y_pred.mean())   # % predicho como 1

    precisions = np.array(precisions)
    recalls   = np.array(recalls)
    f1s       = np.array(f1s)
    queue_rates = np.array(queue_rates)

    # 3) Mejor threshold según F1 (usando TODOS los umbrales)
    best_idx = int(np.nanargmax(f1s))
    best_thr = float(thresholds[best_idx])
    best_f1  = float(f1s[best_idx])

    print(f"Mejor F1 = {best_f1:.4f}, en umbral = {best_thr:.4f}")

    # 4) DataFrame base
    data = {
        "threshold": thresholds,
        "precision": precisions,
        "recall": recalls,
        "f1": f1s,
        "queue_rate": queue_rates
    }
    df = pd.DataFrame(data)

    # FILTRO SOLO PARA EL GRÁFICO:
    # nos quedamos con los umbrales donde aún se predice algún positivo
    mask_plot = df["queue_rate"] > 0
    df_plot = df.loc[mask_plot].copy()

    value_vars = ["precision", "recall", "f1"]
    if include_queue_rate:
        value_vars.append("queue_rate")

    df_long = df_plot.melt(
        id_vars="threshold",
        value_vars=value_vars,
        var_name="métrica",
        value_name="valor"
    )

    # Gráfico
    fig = px.line(
        df_long,
        x="threshold",
        y="valor",
        color="métrica",
        title=title,
        labels={
            "threshold": "Umbral de decisión",
            "valor": "Valor de la métrica"
        }
    )

    fig.add_vline(
        x=best_thr,
        line_dash="dash",
        line_color="black",
        annotation_text=f"F1 máx en umbral = {best_thr:.2f}",
        annotation_position="top left"
    )

    fig.update_layout(
        template="plotly_white",
        height=500,
        width=900,
        legend_title_text=""
    )

    fig.show(renderer='iframe')

    return fig, best_thr, best_f1

In [49]:
fig, best_thr, best_f1 = plot_threshold_curve(model_5, X_test, y_test, include_queue_rate=False)

Mejor F1 = 0.6646, en umbral = 0.3500


In [67]:
from sklearn.model_selection import TunedThresholdClassifierCV

best = grid_search.best_estimator_
tuned = TunedThresholdClassifierCV(best, scoring='f1', cv=3)
tuned.fit(X_train, y_train)
print(tuned.best_threshold_)

0.33380137306033775


### Youden 

Youden (≈ 0,09): recall alto, precisión baja. Útil si perder un cliente es caro y contactar a uno de más es barato.

In [54]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
best_threshold = thresholds[np.argmax(tpr - fpr)]
best_threshold

np.float64(0.08606863975870534)

Parece muy bajo, pero tiene sentido si el churn es poco frecuente en los datos. Si solo un 9 % de los clientes hace churn, el modelo rara vez asigna probabilidades altas.

Si usamos umbral = 0.5 en un problema cuya penetración real es del 10%, estamos diciendo:  
*“Solo consideraré como positivo un caso cuya probabilidad sea al menos 0.50, es decir, 5 veces mayor que la probabilidad media real (que es del 0.10)”*  
  
Como consecuencia el modelo casi nunca predice positivos, por lo que el recall se desploma (no detecta los casos), la matriz de confusión queda desequilibrada y el sistema pierde capacidad operativa. 

In [66]:
y_pred = (y_proba >= 0.086).astype(int)

print(f"{confusion_matrix(y_test, y_pred)}")
print(f"\n {classification_report(y_test, y_pred, digits=3)}")

[[164630  20943]
 [  3389  14940]]

               precision    recall  f1-score   support

         0.0      0.980     0.887     0.931    185573
         1.0      0.416     0.815     0.551     18329

    accuracy                          0.881    203902
   macro avg      0.698     0.851     0.741    203902
weighted avg      0.929     0.881     0.897    203902



**Resumen en una frase:** con este umbral, contactando al 17,6 % de los clientes se captura el 81,5 % del churn, aunque más de la mitad de los contactados no se iba a ir.

Aplicamos el umbral de corte

In [4]:
y_pred = (y_proba >= best_threshold).astype(int)

roc_auc_score(y_test, y_pred)

NameError: name 'y_proba' is not defined

## MlFlow

In [3]:

import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# Configuración de MLflow
mlflow.set_experiment("KKBox_Churn_Prediction")

# Modelo baseline
model = model_5
threshold = 0.5

# Evaluación sobre validación interna
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= threshold).astype(int)

metrics = {
    "roc_auc": roc_auc_score(y_test, y_prob),
    "average_precision": average_precision_score(y_test, y_prob),
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0)
}

# Variables utilizadas
feature_names = list(X_train.columns)

with mlflow.start_run(run_name="baseline_hgb_v1"):

    # Configuración del experimento
    mlflow.log_param("model_type", "HistGradientBoosting")
    mlflow.log_param("threshold", threshold)
    mlflow.log_param("cutoff_date", "2017-03-01")
    mlflow.log_param("feature_version", "v1")
    mlflow.log_param("n_features", len(feature_names))
    mlflow.log_param("selection_criterion", "ROC-AUC vs fit_time")

    # Hiperparámetros del estimador
    mlflow.log_params(
        model.named_steps["model"].get_params(deep=False)
    )

    # Métricas
    mlflow.log_metrics(metrics)

    # Lista de variables
    mlflow.log_text(
        "\n".join(feature_names),
        "features/feature_names.txt"
    )

    # Modelo entrenado
    mlflow.sklearn.log_model(
        sk_model=model,
        name="model"
    )

print("Baseline registrado en MLflow.")


2026/11/08 11:54:03 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/08 11:54:03 INFO mlflow.store.db.utils: Updating database tables
2026/11/08 11:54:04 INFO mlflow.tracking.fluent: Experiment with name 'KKBox_Churn_Prediction' does not exist. Creating a new experiment.


NameError: name 'model_5' is not defined